In [ ]:
import kagglehub
import keras
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
path = kagglehub.competition_download("playground-series-s4e10")
print(path)


train_df = pd.read_csv(path + "/train.csv")

# print column names
print(train_df.columns)

/Users/ignacio/.cache/kagglehub/competitions/playground-series-s4e10
Index(['id', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'loan_status'],
      dtype='str')


In [ ]:
rain_df = train_df.drop(columns=["id"])
train_df = pd.get_dummies(train_df, columns=["person_home_ownership"], drop_first=True)
train_df = pd.get_dummies(train_df, columns=["loan_intent"], drop_first=True)
train_df = pd.get_dummies(train_df, columns=["loan_grade"], drop_first=True)
train_df["cb_person_default_on_file"] = train_df["cb_person_default_on_file"].map(
    {"Y": 1, "N": 0}
)
bool_cols = train_df.select_dtypes(include="bool").columns
train_df[bool_cols] = train_df[bool_cols].astype(int)


X = train_df.drop(columns=["loan_status"])
y = train_df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Further split train into train/val for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

numeric_cols = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
]

scaler = StandardScaler()
X_tr[numeric_cols] = scaler.fit_transform(X_tr[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Class imbalance weight
pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"Positive class weight: {pos_weight:.2f}")


Positive class weight: 5.96


In [ ]:
# keras class
class CustomPredictorModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.layer1 = keras.layers.Dense(32, activation=keras.activations.relu)
        self.layer2 = keras.layers.Dense(5, activation=keras.activations.relu)

    def call(self, inputs):
        output1 = self.layer1(inputs)
        return self.layer2(output1)


model = CustomPredictorModel()